In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from models.nnunet import nnUNet3D
from utils.unets_helper_functions import (
    set_seed,
    train_one_epoch,
    validate_one_epoch,
    save_checkpoint,
    PatchDataset,
    evaluate_full_volume
)

In [3]:
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [4]:
FOLD = 0

with open(f"../data/splits/fold_{FOLD}/train.txt") as f:
    train_cases = f.read().splitlines()

with open(f"../data/splits/fold_{FOLD}/val.txt") as f:
    val_cases = f.read().splitlines()

print("Train cases:", len(train_cases))
print("Val cases:", len(val_cases))


Train cases: 33
Val cases: 9


In [5]:
PATCH_SIZE = 80
PATCHES_PER_CASE = 6

train_dataset = PatchDataset(train_cases,
                        "../data/processed/imagesTr",
                        "../data/processed/labelsTr",
                        patches_per_case = PATCHES_PER_CASE,
                        patch_size = PATCH_SIZE,
                        augment=False)

val_dataset = PatchDataset(val_cases,
                        "../data/processed/imagesTr",
                        "../data/processed/labelsTr",
                        patches_per_case = 1,
                        patch_size = PATCH_SIZE,
                        augment=False)


train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=2,      
    pin_memory=True    
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    num_workers=2,    
    pin_memory=True
)

In [5]:
model = nnUNet3D(
    in_channels=1,
    out_channels=7,
    base_filters=24   # or try 32
).to(device)

print("nnU-Net style model Initialized")


nnU-Net style model Initialized


In [7]:
optimizer = optim.Adam(model.parameters(), lr=1e-4)
weights = torch.tensor([0.1, 1, 1, 1, 1, 1, 1]).to(device)
ce_loss = nn.CrossEntropyLoss(weight=weights)
scaler = torch.cuda.amp.GradScaler()


C:\Users\dhanu\AppData\Local\Temp\ipykernel_14652\2337973256.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [8]:
EPOCHS = 40

PROJECT_ROOT = os.path.abspath("..")
SAVE_DIR = os.path.join(PROJECT_ROOT, "experiments", "nnunetpp_fold0")

os.makedirs(SAVE_DIR, exist_ok=True)
best_val_loss = float("inf")

best_val_loss = float("inf")

for epoch in range(EPOCHS):

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scaler,
        ce_loss,
        device
    )

    val_loss, val_dice = validate_one_epoch(
        model,
        val_loader,
        ce_loss,
        device
    )

    print(f"\nEpoch {epoch}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Per Class Dice: {val_dice}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, "best_model.pth")
        )
        print("Saved Best Model")

    # Save checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, f"checkpoint_epoch_{epoch+1}.pth")
        )
        print("Checkpoint Saved")

100%|██████████| 99/99 [05:21<00:00,  3.25s/it]



Epoch 0
Train Loss: 3.4026
Val Loss: 1.9950
Per Class Dice: [0.13180196 0.0105118  0.00101034 0.00569981 0.00023748 0.12202924]
Saved Best Model


100%|██████████| 99/99 [04:58<00:00,  3.02s/it]



Epoch 1
Train Loss: 1.4702
Val Loss: 1.4050
Per Class Dice: [0.56932905 0.26144846 0.22222291 0.2444134  0.25652573 0.60795788]
Saved Best Model


100%|██████████| 99/99 [04:56<00:00,  3.00s/it]


...


Epoch 37
Train Loss: 0.3750
Val Loss: 0.2699
Per Class Dice: [0.93737793 0.77435604 0.86236962 0.80156297 0.89332428 0.9171428 ]


100%|██████████| 99/99 [04:34<00:00,  2.77s/it]



Epoch 38
Train Loss: 0.3662
Val Loss: 0.4243
Per Class Dice: [0.72609972 0.6748185  0.79153927 0.77875723 0.67026927 0.67113836]


100%|██████████| 99/99 [04:43<00:00,  2.86s/it]



Epoch 39
Train Loss: 0.3902
Val Loss: 0.5050
Per Class Dice: [0.88785268 0.81573706 0.78269518 0.69353323 0.56974679 0.77746505]
Checkpoint Saved


In [9]:
print("Training Complete")

Training Complete


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = nnUNet3D().to(device)

model_path = "../experiments/nnunetpp_fold0/best_model.pth"
checkpoint = torch.load(model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

print("Model loaded for testing.")


C:\Users\dhanu\AppData\Local\Temp\ipykernel_11048\4066954264.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location=device)


Model loaded for testing.


In [7]:
mean_dice, std_dice = evaluate_full_volume(
    model,
    val_cases,
    "../data/processed/imagesTr",
    "../data/processed/labelsTr",
    device=device
)

Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_38 Dice: [np.float64(0.6196456100289877), np.float64(0.7464359176207986), np.float64(0.6768253973383723), np.float64(0.23998545247571426), np.float64(0.7034990794378879), np.float64(0.8106499035770223)]
Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_05 Dice: [np.float64(0.5677193994080234), np.float64(0.7117778982972872), np.float64(0.6650007814071517), np.float64(0.5004158325888138), np.float64(0.6574827255936837), np.float64(0.7488715689937144)]
Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_22 Dice: [np.float64(0.5795595755506203), np.float64(0.7286587288296739), np.float64(0.656856833124042), np.float64(0.3818759216905194), np.float64(0.7979752192520586), np.float64(0.7363871864868405)]
Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_14 Dice: [np.float64(0.5614108357262214), np.float64(0.7647406048634219), np.float64(0.6592125

In [7]:
mean_dice, std_dice = evaluate_full_volume(
    model,
    val_cases,
    "../data/processed/imagesTr",
    "../data/processed/labelsTr",
    device=device
)

Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_38 Dice: [np.float64(0.6808510638894568), np.float64(0.7702923811835438), np.float64(0.7092841517516659), np.float64(0.3824827244265745), np.float64(0.6402521389462313), np.float64(0.8052717311145217)]
Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_05 Dice: [np.float64(0.5950999356022489), np.float64(0.7125377645675697), np.float64(0.6398880340939286), np.float64(0.5541703122068495), np.float64(0.7251303845252413), np.float64(0.7221948116385527)]
Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_22 Dice: [np.float64(0.6441575785345742), np.float64(0.7759548835068221), np.float64(0.6677986662610083), np.float64(0.4519861425946029), np.float64(0.7449795629731772), np.float64(0.7120007755292749)]
Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_14 Dice: [np.float64(0.606482056457949), np.float64(0.7783436825658664), np.float64(0.62318628